# SyntaxLM - Decoder-Only Transformer From Scratch

> **Goal:** Train a modern decoder-only LLM (RoPE · GQA · SwiGLU · RMSNorm) from scratch on Google Colab.  
> **Model size:** ~15M parameters   
> **Stack:** PyTorch · SentencePiece · FineWeb · Gradio · HuggingFace Hub

---
| Phase | Topic |
|-------|-------|
| 0 | Environment Setup |
| 1 | Load FineWeb Dataset |
| 2 | Text Cleaning |
| 3 | Train SentencePiece Tokenizer |
| 4 | Tokenization |
| 5 | Transformer Architecture |
| 5.5 | Sanity Check (overfit test) |
| 6 | Training Pipeline |
| 7 | Save Model |
| 8 | Text Generation |
| 9 | Gradio Demo |

## Phase 0 Environment Setup


In [1]:
# Install dependencies
!pip install -q sentencepiece datasets gradio huggingface_hub
!nvidia-smi

Thu May 28 13:24:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
#Imports
import os, math, json, time, unicodedata, re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import sentencepiece as spm
from pathlib import Path

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Paths
os.makedirs('syntaxlm', exist_ok=True)
CORPUS_PATH   = 'syntaxlm/corpus.txt'
MODEL_PREFIX  = 'syntaxlm/syntaxlm'
TOKENS_PATH   = 'syntaxlm/tokens.npy'
CKPT_PATH     = 'syntaxlm/syntaxlm_best.pt'
CONFIG_PATH   = 'syntaxlm/config.json'
print('Paths initialised')

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB
Paths initialised ✓


## Phase 1 Load FineWeb Dataset
Stream a small, clean slice (~30 MB text) to stay within Colab RAM limits.

In [18]:
from datasets import load_dataset

# Config
MAX_SAMPLES   = 50_000      # hard cap on documents read
MAX_CHARS     = 30_000_000  # ~30 MB of raw text
MIN_DOC_LEN   = 200         # skip very short docs

print('Streaming FineWeb sample-10BT …')
dataset = load_dataset(
    'HuggingFaceFW/fineweb',
    name='sample-10BT',
    split='train',
    streaming=True,
)

total_chars = 0
kept = 0

with open(CORPUS_PATH, 'w', encoding='utf-8') as f:
    for i, doc in enumerate(dataset):
        if i >= MAX_SAMPLES or total_chars >= MAX_CHARS:
            break
        text = doc.get('text', '').strip()
        if len(text) < MIN_DOC_LEN:
            continue
        f.write(text + '\n')
        total_chars += len(text)
        kept += 1
        if kept % 5000 == 0:
            print(f'  {kept:,} docs | {total_chars/1e6:.1f} MB')

print(f'\nDone: {kept:,} documents | {total_chars/1e6:.2f} MB → {CORPUS_PATH}')

Streaming FineWeb sample-10BT …


Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

  5,000 docs | 15.1 MB

Done: 9,830 documents | 30.00 MB → syntaxlm/corpus.txt


## Phase 2 Text Cleaning
Remove noise: broken unicode, duplicate whitespace, near-empty lines.

In [4]:
def clean_text(text: str) -> str:
    """Normalise unicode, strip control chars, collapse whitespace."""
    # NFKC normalisation (e.g. ﬁ → fi, ² → 2)
    text = unicodedata.normalize('NFKC', text)
    # Remove non-printable / control characters except newline and tab
    text = re.sub(r'[\x00-\x08\x0b-\x1f\x7f-\x9f]', '', text)
    # Collapse multiple spaces / tabs
    text = re.sub(r'[ \t]+', ' ', text)
    # Collapse 3+ newlines → 2
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

CLEAN_PATH = CORPUS_PATH.replace('.txt', '_clean.txt')
kept_lines = 0
dropped    = 0

with open(CORPUS_PATH, 'r', encoding='utf-8', errors='ignore') as fin, \
     open(CLEAN_PATH,  'w', encoding='utf-8') as fout:
    for line in fin:
        cleaned = clean_text(line)
        if len(cleaned) < 80:
            dropped += 1
            continue
        fout.write(cleaned + '\n')
        kept_lines += 1

size_mb = os.path.getsize(CLEAN_PATH) / 1e6
print(f'Kept  : {kept_lines:,} lines')
print(f'Dropped: {dropped:,} lines')
print(f'Clean corpus: {size_mb:.2f} MB → {CLEAN_PATH}')

Kept  : 91,943 lines
Dropped: 73,289 lines
Clean corpus: 27.02 MB → syntaxlm/corpus_clean.txt


## Phase 3 Train SentencePiece BPE Tokenizer
Vocabulary size 8 000 small enough for a 15M model, large enough to be useful.

In [5]:
VOCAB_SIZE = 8_000

spm.SentencePieceTrainer.train(
    input=CLEAN_PATH,
    model_prefix=MODEL_PREFIX,
    vocab_size=VOCAB_SIZE,
    model_type='bpe',
    character_coverage=0.9995,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
    pad_piece='<pad>',
    unk_piece='<unk>',
    bos_piece='<bos>',
    eos_piece='<eos>',
    num_threads=4,
    input_sentence_size=500_000,
    shuffle_input_sentence=True,
)

# Verify
sp = spm.SentencePieceProcessor(model_file=f'{MODEL_PREFIX}.model')
sample = 'The transformer architecture uses attention mechanisms.'
ids    = sp.encode(sample)
print(f'Tokenizer trained  vocab={sp.vocab_size()}')
print(f'Sample : "{sample}"')
print(f'Tokens : {ids}')
print(f'Decoded: "{sp.decode(ids)}"')

Tokenizer trained  ✓  vocab=8000
Sample : "The transformer architecture uses attention mechanisms."
Tokens : [96, 4966, 11, 7523, 3150, 2503, 5561, 7911, 7925]
Decoded: "The transformer architecture uses attention mechanisms."


## Phase 4 Tokenize Corpus
Encode clean text → token IDs → save as `tokens.npy` (int16 to halve disk usage).

In [6]:
# Tokenise line-by-line and flatten to 1-D array
MAX_TOKENS = 20_000_000   # ~20M tokens → tokens.npy ≈ 40 MB (int16)

sp = spm.SentencePieceProcessor(model_file=f'{MODEL_PREFIX}.model')
BOS, EOS = sp.bos_id(), sp.eos_id()

all_ids = []
total   = 0

with open(CLEAN_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        if total >= MAX_TOKENS:
            break
        line = line.strip()
        if not line:
            continue
        ids = [BOS] + sp.encode(line) + [EOS]
        all_ids.extend(ids)
        total += len(ids)

# int16 can hold values 0-32767 → fine for vocab_size=8000
tokens_np = np.array(all_ids[:MAX_TOKENS], dtype=np.int16)
np.save(TOKENS_PATH, tokens_np)

print(f'Total tokens : {len(tokens_np):,}')
print(f'File size    : {os.path.getsize(TOKENS_PATH)/1e6:.1f} MB')
print(f'Saved → {TOKENS_PATH}')

Total tokens : 7,056,913
File size    : 14.1 MB
Saved → syntaxlm/tokens.npy


## Phase 5 Transformer Architecture

Modern decoder-only LLM with:
- **RMSNorm** — faster than LayerNorm, used in LLaMA
- **RoPE** — Rotary Position Embeddings (no absolute position table)
- **Grouped Query Attention (GQA)** — fewer KV heads → less memory
- **SwiGLU** — gated activation (better than ReLU/GELU in practice)
- **Weight Tying** — embedding and LM-head share weights


In [7]:
# Model Config — ~15M parameters
CFG = dict(
    vocab_size  = VOCAB_SIZE,   # 8 000
    d_model     = 256,          # embedding dimension
    n_heads     = 8,            # query heads
    n_kv_heads  = 2,            # key/value heads (GQA: groups = n_heads // n_kv_heads = 4)
    n_layers    = 6,            # transformer blocks
    ffn_mult    = 2.67,         # hidden = int(d_model * ffn_mult), rounded to mult of 64
    max_seq_len = 256,          # context window
    dropout     = 0.1,
)

# Save config for later loading
with open(CONFIG_PATH, 'w') as f:
    json.dump(CFG, f, indent=2)
print('Config saved:', CFG)

Config saved: {'vocab_size': 8000, 'd_model': 256, 'n_heads': 8, 'n_kv_heads': 2, 'n_layers': 6, 'ffn_mult': 2.67, 'max_seq_len': 256, 'dropout': 0.1}


In [8]:
#  RMSNorm
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalisation (no mean subtraction)."""
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps   = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, D)
        rms = x.float().pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return (x.float() / rms * self.scale).to(x.dtype)


#  Rotary Position Embeddings (RoPE)
def precompute_rope(head_dim: int, max_seq: int, base: float = 10_000.0, device=None):
    """Returns (cos, sin) tensors of shape (max_seq, head_dim//2)."""
    half = head_dim // 2
    theta = 1.0 / (base ** (torch.arange(0, half, device=device).float() / half))
    pos   = torch.arange(max_seq, device=device).float()
    freqs = torch.outer(pos, theta)           # (max_seq, half)
    return freqs.cos(), freqs.sin()


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """Apply RoPE to query or key tensor x of shape (B, n_heads, T, head_dim)."""
    # x: (..., T, head_dim)
    T = x.shape[-2]
    cos = cos[:T].unsqueeze(0).unsqueeze(0)   # (1, 1, T, half)
    sin = sin[:T].unsqueeze(0).unsqueeze(0)
    half = x.shape[-1] // 2
    x1, x2 = x[..., :half], x[..., half:]    # split along head_dim
    rotated = torch.cat([-x2, x1], dim=-1)
    cos_full = cos.repeat(1, 1, 1, 2)         # broadcast to full head_dim
    sin_full = sin.repeat(1, 1, 1, 2)
    return x * cos_full + rotated * sin_full


#  Grouped Query Attention
class GQAttention(nn.Module):
    def __init__(self, cfg: dict):
        super().__init__()
        self.n_heads    = cfg['n_heads']
        self.n_kv_heads = cfg['n_kv_heads']
        self.groups     = self.n_heads // self.n_kv_heads
        self.head_dim   = cfg['d_model'] // self.n_heads
        D = cfg['d_model']
        KV = self.n_kv_heads * self.head_dim

        self.q_proj  = nn.Linear(D, D,  bias=False)
        self.k_proj  = nn.Linear(D, KV, bias=False)
        self.v_proj  = nn.Linear(D, KV, bias=False)
        self.o_proj  = nn.Linear(D, D,  bias=False)
        self.drop    = nn.Dropout(cfg['dropout'])

        self.register_buffer('mask', torch.tril(
            torch.ones(cfg['max_seq_len'], cfg['max_seq_len'])
        ).view(1, 1, cfg['max_seq_len'], cfg['max_seq_len']))

    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        B, T, D = x.shape
        H, KVH, Hd = self.n_heads, self.n_kv_heads, self.head_dim

        q = self.q_proj(x).view(B, T, H,   Hd).transpose(1, 2)   # (B, H, T, Hd)
        k = self.k_proj(x).view(B, T, KVH, Hd).transpose(1, 2)   # (B, KVH, T, Hd)
        v = self.v_proj(x).view(B, T, KVH, Hd).transpose(1, 2)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        # Expand KV heads to match query heads (GQA)
        k = k.unsqueeze(2).expand(-1, -1, self.groups, -1, -1).reshape(B, H, T, Hd)
        v = v.unsqueeze(2).expand(-1, -1, self.groups, -1, -1).reshape(B, H, T, Hd)

        scale  = math.sqrt(Hd)
        scores = torch.matmul(q, k.transpose(-2, -1)) / scale      # (B, H, T, T)
        scores = scores.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        attn   = F.softmax(scores, dim=-1)
        attn   = self.drop(attn)

        out = torch.matmul(attn, v).transpose(1, 2).reshape(B, T, D)
        return self.o_proj(out)


#  SwiGLU Feed-Forward
class SwiGLUFFN(nn.Module):
    """SwiGLU: out = (xW1 * silu(xW2)) @ W3"""
    def __init__(self, cfg: dict):
        super().__init__()
        hidden = int(cfg['d_model'] * cfg['ffn_mult'])
        hidden = (hidden + 63) // 64 * 64          # round up to multiple of 64
        D = cfg['d_model']
        self.w1   = nn.Linear(D, hidden, bias=False)
        self.w2   = nn.Linear(D, hidden, bias=False)
        self.w3   = nn.Linear(hidden, D, bias=False)
        self.drop = nn.Dropout(cfg['dropout'])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.drop(self.w3(self.w1(x) * F.silu(self.w2(x))))


#  Transformer Block
class Block(nn.Module):
    def __init__(self, cfg: dict):
        super().__init__()
        self.attn_norm = RMSNorm(cfg['d_model'])
        self.attn      = GQAttention(cfg)
        self.ffn_norm  = RMSNorm(cfg['d_model'])
        self.ffn       = SwiGLUFFN(cfg)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.attn_norm(x), cos, sin)   # pre-norm + residual
        x = x + self.ffn(self.ffn_norm(x))
        return x


#  SyntaxLM Model
class SyntaxLM(nn.Module):
    def __init__(self, cfg: dict):
        super().__init__()
        self.cfg = cfg
        self.embed  = nn.Embedding(cfg['vocab_size'], cfg['d_model'])
        self.drop   = nn.Dropout(cfg['dropout'])
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg['n_layers'])])
        self.norm   = RMSNorm(cfg['d_model'])
        self.lm_head = nn.Linear(cfg['d_model'], cfg['vocab_size'], bias=False)

        # Weight tying embedding and lm_head share parameters
        self.lm_head.weight = self.embed.weight

        # Precompute RoPE buffers
        head_dim = cfg['d_model'] // cfg['n_heads']
        cos, sin = precompute_rope(head_dim, cfg['max_seq_len'])
        self.register_buffer('cos', cos)
        self.register_buffer('sin', sin)

        # Initialise weights
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor = None):
        B, T = idx.shape
        assert T <= self.cfg['max_seq_len'], f'Sequence too long: {T}'

        x = self.drop(self.embed(idx))                    # (B, T, D)
        cos, sin = self.cos[:T], self.sin[:T]

        for block in self.blocks:
            x = block(x, cos, sin)

        x      = self.norm(x)
        logits = self.lm_head(x)                          # (B, T, V)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, self.cfg['vocab_size']),
                targets.view(-1),
                ignore_index=0,   # pad token
            )
        return logits, loss

    def num_params(self) -> int:
        return sum(p.numel() for p in self.parameters())


# Instantiate and verify
model = SyntaxLM(CFG).to(DEVICE)
total = model.num_params()
print(f'SyntaxLM parameters: {total/1e6:.2f}M')

# Quick forward-pass smoke test
dummy_x = torch.randint(0, CFG['vocab_size'], (2, 32)).to(DEVICE)
dummy_y = torch.randint(0, CFG['vocab_size'], (2, 32)).to(DEVICE)
logits, loss = model(dummy_x, dummy_y)
print(f'Forward pass OK — logits: {logits.shape} | loss: {loss.item():.4f}')

SyntaxLM parameters: 6.28M
Forward pass OK — logits: torch.Size([2, 32, 8000]) | loss: 9.0826


## Phase 5.5 Sanity Check (Overfit Test)

Before full training, **overfit on 64 samples**.  
If loss drops to < 1.0 the model and data pipeline are working correctly.

In [9]:
SEQ = CFG['max_seq_len']

# Load a tiny slice of tokens
tokens_all = np.load(TOKENS_PATH).astype(np.int64)
tiny = torch.from_numpy(
    tokens_all[:64 * SEQ + 1]
).to(DEVICE)

x_tiny = tiny[:-1].view(64, SEQ)
y_tiny = tiny[1:].view(64, SEQ)

opt_sanity = torch.optim.AdamW(model.parameters(), lr=3e-3)
model.train()

print('Overfit sanity check …')
for step in range(200):
    opt_sanity.zero_grad()
    _, loss = model(x_tiny, y_tiny)
    loss.backward()
    opt_sanity.step()
    if step % 50 == 0:
        print(f'  step {step:3d}  loss={loss.item():.4f}')

print()
if loss.item() < 2.0:
    print('Model can overfit — training pipeline is healthy!')
else:
    print('Loss still high — check data pipeline or model config.')

# Re-init weights before real training
model.apply(model._init_weights)
print('Weights re-initialised for full training.')

Overfit sanity check …
  step   0  loss=9.0359
  step  50  loss=6.2647
  step 100  loss=3.5888
  step 150  loss=0.9275

✓ Model can overfit — training pipeline is healthy!
Weights re-initialised for full training.


## Phase 6 Training Pipeline
Mixed precision · gradient accumulation · cosine LR schedule · checkpoint saving.

In [10]:
#  Dataset — sliding window over the flat token array
class TokenDataset(Dataset):
    def __init__(self, tokens: np.ndarray, seq_len: int):
        self.tokens  = torch.from_numpy(tokens.astype(np.int64))
        self.seq_len = seq_len
        self.n       = (len(tokens) - 1) // seq_len

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        s = idx * self.seq_len
        x = self.tokens[s     : s + self.seq_len]
        y = self.tokens[s + 1 : s + self.seq_len + 1]
        return x, y


# Training Hyperparameters
TRAIN_CFG = dict(
    batch_size      = 32,
    grad_accum      = 4,          # effective batch = 32 * 4 = 128
    max_steps       = 5_000,
    lr              = 3e-4,
    warmup_steps    = 200,
    weight_decay    = 0.1,
    grad_clip       = 1.0,
    eval_every      = 250,
    save_every      = 500,
    val_frac        = 0.02,       # 2% of data for validation
)

# Build Dataset & DataLoaders
tokens_all = np.load(TOKENS_PATH)
n_val      = int(len(tokens_all) * TRAIN_CFG['val_frac'])
val_tokens = tokens_all[:n_val]
trn_tokens = tokens_all[n_val:]

trn_ds = TokenDataset(trn_tokens, CFG['max_seq_len'])
val_ds = TokenDataset(val_tokens, CFG['max_seq_len'])

trn_loader = DataLoader(trn_ds, batch_size=TRAIN_CFG['batch_size'],
                        shuffle=True,  num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=TRAIN_CFG['batch_size'],
                        shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(trn_ds):,} samples | Val: {len(val_ds):,} samples')


# Cosine LR with Linear Warmup
def get_lr(step: int, cfg: dict) -> float:
    if step < cfg['warmup_steps']:
        return cfg['lr'] * step / cfg['warmup_steps']
    progress = (step - cfg['warmup_steps']) / max(1, cfg['max_steps'] - cfg['warmup_steps'])
    return cfg['lr'] * 0.5 * (1.0 + math.cos(math.pi * progress))


@torch.no_grad()
def evaluate(model, loader, max_batches=30) -> float:
    model.eval()
    total, count = 0.0, 0
    for i, (x, y) in enumerate(loader):
        if i >= max_batches:
            break
        x, y = x.to(DEVICE), y.to(DEVICE)
        with autocast():
            _, loss = model(x, y)
        total += loss.item()
        count += 1
    model.train()
    return total / max(count, 1)

Train: 27,014 samples | Val: 551 samples


In [11]:
#  Training Loop
model      = SyntaxLM(CFG).to(DEVICE)
optimizer  = torch.optim.AdamW(
    model.parameters(),
    lr=TRAIN_CFG['lr'],
    weight_decay=TRAIN_CFG['weight_decay'],
    betas=(0.9, 0.95),
)
scaler     = GradScaler()            # mixed precision
best_val   = float('inf')
step       = 0
accum_loss = 0.0
t0         = time.time()
model.train()

print(f'Starting training — {TRAIN_CFG["max_steps"]:,} steps')
print(f'Params: {model.num_params()/1e6:.2f}M | Device: {DEVICE}\n')

while step < TRAIN_CFG['max_steps']:
    for x, y in trn_loader:
        if step >= TRAIN_CFG['max_steps']:
            break

        # Update LR
        lr = get_lr(step, TRAIN_CFG)
        for g in optimizer.param_groups:
            g['lr'] = lr

        x, y = x.to(DEVICE), y.to(DEVICE)

        # Forward with mixed precision
        with autocast():
            _, loss = model(x, y)
            loss    = loss / TRAIN_CFG['grad_accum']

        scaler.scale(loss).backward()
        accum_loss += loss.item()

        # Gradient accumulation
        if (step + 1) % TRAIN_CFG['grad_accum'] == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), TRAIN_CFG['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        step += 1

        # Logging
        if step % 50 == 0:
            elapsed = time.time() - t0
            tok_s   = step * TRAIN_CFG['batch_size'] * CFG['max_seq_len'] / elapsed
            print(f'step {step:5d} | loss {accum_loss*TRAIN_CFG["grad_accum"]/50:.4f} '
                  f'| lr {lr:.2e} | {tok_s/1e3:.1f}k tok/s')
            accum_loss = 0.0

        # Validation
        if step % TRAIN_CFG['eval_every'] == 0:
            val_loss = evaluate(model, val_loader)
            val_ppl  = math.exp(min(val_loss, 20))
            print(f'  val_loss={val_loss:.4f}  val_ppl={val_ppl:.1f}')
            if val_loss < best_val:
                best_val = val_loss
                torch.save(model.state_dict(), CKPT_PATH)
                print(f' Best checkpoint saved (val={best_val:.4f})')

total_time = (time.time() - t0) / 60
print(f'\nTraining complete in {total_time:.1f} min')
print(f'Best val loss: {best_val:.4f} | perplexity: {math.exp(min(best_val,20)):.1f}')

Starting training — 5,000 steps
Params: 6.28M | Device: cuda



/tmp/ipykernel_1335/1972449874.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler     = GradScaler()            # mixed precision
/tmp/ipykernel_1335/1972449874.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


step    50 | loss 8.9416 | lr 7.35e-05 | 65.0k tok/s
step   100 | loss 8.5682 | lr 1.48e-04 | 67.5k tok/s
step   150 | loss 8.1820 | lr 2.23e-04 | 68.8k tok/s
step   200 | loss 7.7100 | lr 2.98e-04 | 69.6k tok/s
step   250 | loss 7.3249 | lr 3.00e-04 | 70.2k tok/s


/tmp/ipykernel_1335/4022165942.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  ↳ val_loss=7.2604  val_ppl=1422.8
  ✓ Best checkpoint saved (val=7.2604)
step   300 | loss 7.1440 | lr 3.00e-04 | 68.9k tok/s
step   350 | loss 7.0362 | lr 2.99e-04 | 69.5k tok/s
step   400 | loss 6.9094 | lr 2.99e-04 | 69.8k tok/s
step   450 | loss 6.7926 | lr 2.98e-04 | 70.2k tok/s
step   500 | loss 6.6444 | lr 2.97e-04 | 70.4k tok/s
  ↳ val_loss=6.6425  val_ppl=767.0
  ✓ Best checkpoint saved (val=6.6425)
step   550 | loss 6.5380 | lr 2.96e-04 | 69.6k tok/s
step   600 | loss 6.4210 | lr 2.95e-04 | 69.8k tok/s
step   650 | loss 6.3457 | lr 2.94e-04 | 70.0k tok/s
step   700 | loss 6.2712 | lr 2.92e-04 | 70.1k tok/s
step   750 | loss 6.2026 | lr 2.90e-04 | 70.2k tok/s
  ↳ val_loss=6.2149  val_ppl=500.1
  ✓ Best checkpoint saved (val=6.2149)
step   800 | loss 6.1425 | lr 2.89e-04 | 69.7k tok/s
step   850 | loss 6.0800 | lr 2.87e-04 | 69.7k tok/s
step   900 | loss 6.0278 | lr 2.85e-04 | 69.8k tok/s
step   950 | loss 5.9937 | lr 2.82e-04 | 69.9k tok/s
step  1000 | loss 5.9538 | lr 2.80e

## Phase 7 Save Model Artifacts
Checkpoint, tokenizer, and config for reproducible loading.

In [12]:
# All primary artifacts are already saved during training.
# This confirms them and prints sizes.

artifacts = [
    CKPT_PATH,
    f'{MODEL_PREFIX}.model',
    f'{MODEL_PREFIX}.vocab',
    CONFIG_PATH,
]

print('SyntaxLM Artifacts')
for path in artifacts:
    if os.path.exists(path):
        size = os.path.getsize(path) / 1e6
        print(f'   {path:<40} {size:.2f} MB')
    else:
        print(f'   {path}  (MISSING)')

# Reload from checkpoint to verify
with open(CONFIG_PATH) as f:
    loaded_cfg = json.load(f)

loaded_model = SyntaxLM(loaded_cfg)
loaded_model.load_state_dict(torch.load(CKPT_PATH, map_location='cpu'))
loaded_model.eval()
print('\nCheckpoint reload ')

── SyntaxLM Artifacts ─────────────────────────────────
  ✓  syntaxlm/syntaxlm_best.pt                26.74 MB
  ✓  syntaxlm/syntaxlm.model                  0.37 MB
  ✓  syntaxlm/syntaxlm.vocab                  0.11 MB
  ✓  syntaxlm/config.json                     0.00 MB

Checkpoint reload ✓


## Phase 8 Text Generation
Autoregressive sampling with temperature and top-k filtering.

In [13]:
@torch.no_grad()
def generate(model, sp, prompt, max_new=200, temperature=0.6,
             top_k=20, rep_penalty=1.3):

    model.eval()
    EOS     = sp.eos_id()
    SEQ_MAX = model.cfg['max_seq_len']

    ids     = [sp.bos_id()] + sp.encode(prompt)
    ids     = ids[-SEQ_MAX:]
    input_t = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    generated = []

    for _ in range(max_new):
        ctx       = input_t[:, -SEQ_MAX:]
        logits, _ = model(ctx)
        logits    = logits[0, -1, :] / temperature

        # Repetition penalty
        # Divide logits of already-seen tokens to discourage repeats
        seen = set(input_t[0].tolist())
        for tok_id in seen:
            if logits[tok_id] > 0:
                logits[tok_id] /= rep_penalty
            else:
                logits[tok_id] *= rep_penalty

        if top_k > 0:
            kth_val = torch.topk(logits, top_k).values[-1]
            logits  = logits.masked_fill(logits < kth_val, float('-inf'))

        probs  = F.softmax(logits, dim=-1)
        next_t = torch.multinomial(probs, num_samples=1)

        if next_t.item() == EOS:
            break

        generated.append(next_t.item())
        input_t = torch.cat([input_t, next_t.unsqueeze(0)], dim=1)

    return sp.decode(generated)


# Test with better settings
prompts = [
    'The history of artificial intelligence',
    'In machine learning, a neural network',
    'The transformer architecture was introduced',
]
for p in prompts:
    out = generate(model, sp, p,
                   max_new=100,
                   temperature=0.6,
                   top_k=20,
                   rep_penalty=1.3)
    print(f'Prompt : {p}')
    print(f'Output : {out}')
    print('-' * 60)

Prompt : The history of artificial intelligence
Output : and the University of EVU, which is a “wo-in-the-of-creative in the United States" (Axk) was to be the same as a man.
------------------------------------------------------------
Prompt : In machine learning, a neural network
Output : of the data is not a very well-piece and it can be used to ensure that its business may have been in some time.
------------------------------------------------------------
Prompt : The transformer architecture was introduced
Output : in the first time, with a new state that has been made up to its company’s business. But I’m not sure this is what we have done.
------------------------------------------------------------


---
##  Project Complete

**What i built end-to-end:**

| Skill | Demonstrated by |
|-------|-----------------|
| Data engineering | FineWeb streaming, cleaning, BPE tokenization |
| Deep learning | RMSNorm, RoPE, GQA, SwiGLU from scratch in PyTorch |
| Training engineering | Mixed precision, grad accum, cosine LR, checkpointing |
| Inference | Temperature + top-k sampling, sliding context window |
| Deployment | Gradio UI + HuggingFace Hub |

